In [39]:
from pydantic import BaseModel
from typing import Literal
import random, json
from enum import Enum
from common.ko_util import korean_to_english_pronunciation

class ReadingQuest(BaseModel):
    quest_type: Literal["symbol","color"]
    quest_level: Literal[1,2,3]
    quest_words: list[str]
    quest_codes: list[str]
class ListeningQuest(BaseModel):
    quest_type: Literal["symbol","color"]
    quest_level: Literal[1,2,3]
    quest_words: list[str]
    quest_codes: list[str]

class StageType(Enum):
    READING = 1
    LISTENING = 2
    WRITING = 3
    SPEAKING = 4
    
quest_template = {
    'word_data1':"{} 스티커를 찾아라",
    'word_data2':"{} 캐리어를 찾아라",
    'full_data':"{} 스티커가 붙은 {} 캐리어를 찾아라"
}
class QuestLevel(Enum):
    EASY = 1
    NORMAL = 2
    HARD = 3
class WordData(BaseModel):
    kor: str
    eng: str
    pronunciation: str
class TargetItem(BaseModel):
    name:str
    code:str
class ReadTargetData(BaseModel):
    symbol:TargetItem
    color:TargetItem
class QuestInfo(BaseModel):
    index:int
    dificulity:QuestLevel
class QuestReadInfo(QuestInfo):
    target_data: list[ReadTargetData]
    correct_answer_index: int
    word_data1: WordData
    word_data2: WordData
    full_data: WordData

In [12]:
stage_data = [
  {
    "quest_type": "symbol",
    "quest_level": 1,
    "quest_words": [
      "개","고양이","새","물고기",
      "소","닭","돼지","토끼"
    ],
    "quest_codes": [
      "1","2","3","4","5","6","7","8"
    ]
  },
  {
    "quest_type": "symbol",
    "quest_level": 2,
    "quest_words": [
      "말","양","사슴","호랑이",
      "곰","여우","원숭이","펭귄"
    ],
    "quest_codes": [
      "9","10","11","12","13","14","15","16"
    ]
  },
  {
    "quest_type": "symbol",
    "quest_level": 3,
    "quest_words": [
      "기린","코끼리","치타","물개",
      "하마","낙타","돌고래","박쥐"
    ],
    "quest_codes": [
      "17","18","19","20","21","22","23","24"
    ]
  },
  {
    "quest_type": "color",
    "quest_level": 1,
    "quest_words":["빨강","파랑","노랑","초록",
      "검정","흰색","회색","주황"],
    "quest_codes": [
      "1","2","3","4","5","6","7","8"
    ]
  },
  {
    "quest_type": "color",
    "quest_level": 2,
    "quest_words":["분홍","갈색","남색","보라색",
      "금색","은색","살구색","하늘색"],
    "quest_codes": [
      "9","10","11","12","13","14","15","16"
    ]
  },
  {
    "quest_type": "color",
    "quest_level": 3,
    "quest_words":["자주색","청록색","황토색","진홍색",
      "군청색","연두색","와인색","베이지"],
    "quest_codes": [
      "17","18","19","20","21","22","23","24"
    ]
  }
]

In [13]:
ReadingQuest(**stage_data[0])

ReadingQuest(quest_type='symbol', quest_level=1, quest_words=['개', '고양이', '새', '물고기', '소', '닭', '돼지', '토끼'], quest_codes=['1', '2', '3', '4', '5', '6', '7', '8'])

In [16]:
def dict_to_quest(stage_data:list[dict], quest_type:StageType):
     quests = []
     for stage in stage_data:
          if quest_type == StageType.READING:
               quests.append(ReadingQuest(**stage))
          elif quest_type == StageType.LISTENING:
               quests.append(ListeningQuest(**stage))
     return quests

In [52]:
def quest_items(quests:list[BaseModel],_type:str,level:QuestLevel):
    items = []
    for item_zip in [zip(q.quest_codes,q.quest_words)
                 for q in quests if q.quest_type == _type and q.quest_level == level]:
        for item in item_zip:
            items.append(TargetItem(code=item[0],name=item[1]))
    return items
def gen_read_quest(quests:list[BaseModel],level:QuestLevel,quest_count:int = 10):
    """
        quests : read quest list
        level  : quest level
        quest_count : 필요 갯수
        읽기 시나리오 생성
    """
    symbols = quest_items(quests,'symbol',level)
    colors = quest_items(quests,'color',level)
    quest_data = random.sample([(symbol, color) for symbol in symbols for color in colors],quest_count)
    correct_index = random.randint(0,quest_count-1)
    target_data = [ReadTargetData(symbol=q_data[0],color=q_data[1]) for q_data in quest_data]
    word_data1 = quest_template['word_data1'].format(quest_data[correct_index][0].name)
    word_data2 = quest_template['word_data2'].format(quest_data[correct_index][1].name)
    full_data = quest_template['full_data'].format(quest_data[correct_index][0].name,quest_data[correct_index][1].name)
    return QuestReadInfo(
        index=1,
        dificulity=QuestLevel.EASY,
        target_data=target_data,
        correct_answer_index=correct_index,
        word_data1=WordData(
            kor = word_data1,
            eng = '',#ko_to_en(word_data1),
            pronunciation=korean_to_english_pronunciation(word_data1)
        ),
        word_data2=WordData(
            kor = word_data2,
            eng = '',#ko_to_en(word_data2),
            pronunciation=korean_to_english_pronunciation(word_data2)
        ),
        full_data=WordData(
            kor = full_data,
            eng = '',#ko_to_en(full_data),
            pronunciation=korean_to_english_pronunciation(full_data)
        )
    )

In [54]:
quests = dict_to_quest(stage_data,StageType.READING)
quest_items(quests,'symbol',2)

[TargetItem(name='말', code='9'),
 TargetItem(name='양', code='10'),
 TargetItem(name='사슴', code='11'),
 TargetItem(name='호랑이', code='12'),
 TargetItem(name='곰', code='13'),
 TargetItem(name='여우', code='14'),
 TargetItem(name='원숭이', code='15'),
 TargetItem(name='펭귄', code='16')]

In [46]:
target_data = gen_read_quest(quests,3,10)
target_data.word_data1

WordData(kor='하마 스티커를 찾아라', eng='', pronunciation='ha-ma seu-ti-keo-reul chat-a-ra')

In [49]:
target_data.target_data

[ReadTargetData(symbol=TargetItem(name='낙타', code='22'), color=TargetItem(name='와인색', code='23')),
 ReadTargetData(symbol=TargetItem(name='물개', code='20'), color=TargetItem(name='황토색', code='19')),
 ReadTargetData(symbol=TargetItem(name='박쥐', code='24'), color=TargetItem(name='와인색', code='23')),
 ReadTargetData(symbol=TargetItem(name='하마', code='21'), color=TargetItem(name='황토색', code='19')),
 ReadTargetData(symbol=TargetItem(name='기린', code='17'), color=TargetItem(name='군청색', code='21')),
 ReadTargetData(symbol=TargetItem(name='하마', code='21'), color=TargetItem(name='자주색', code='17')),
 ReadTargetData(symbol=TargetItem(name='기린', code='17'), color=TargetItem(name='진홍색', code='20')),
 ReadTargetData(symbol=TargetItem(name='치타', code='19'), color=TargetItem(name='청록색', code='18')),
 ReadTargetData(symbol=TargetItem(name='돌고래', code='23'), color=TargetItem(name='자주색', code='17')),
 ReadTargetData(symbol=TargetItem(name='하마', code='21'), color=TargetItem(name='청록색', code='18'))]

In [50]:
target_data.correct_answer_index

9

In [51]:
target_data.full_data

WordData(kor='하마 스티커가 붙은 청록색 캐리어를 찾아라', eng='', pronunciation='ha-ma seu-ti-keo-ga but-eun cheong-rok-saek kae-ri-eo-reul chat-a-ra')